In [1]:
! pip install implicit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 50.4 MB/s eta 0:00:0000:0100:01


In [18]:
import polars as pl
import numpy as np
import sys
sys.path.append("/kaggle/input/models/fedorfesenko/metrcs/other/default/1/")
from metrics import evaluate_candidates

from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix
import pickle
import json
from pathlib import Path

In [19]:
TOP_K = 500

In [20]:
df_train = pl.read_parquet("/kaggle/input/datasets/fedorfesenko/train-interactions/df_interactions_train.parquet")
df_val = pl.read_parquet("/kaggle/input/datasets/fedorfesenko/train-interactions/df_interactions_val.parquet")

In [21]:
cart_train = (
    df_train.filter(pl.col("action_type") == "AT_CartUpdate")
    .select(["user_id", "product_id"])
    .unique()
)
cart_train

user_id,product_id
u64,u64
17704382986039710673,13799421684173424865
1289903229404989439,3632482440523322274
5461952193868252257,1675406005603177042
17169491437553305621,13670405258728010682
1503691436145582357,17697123624618342229
…,…
7545778242401643821,3801324023051417832
874933781112491160,8981166967369157972
11265326036325739430,745711118602346248


In [22]:
cart_val = (
    df_val.filter(pl.col("action_type") == "AT_CartUpdate")
    .group_by("user_id")
    .agg(pl.col("product_id").alias("items"))
)

true_items_val = {
    row["user_id"]: row["items"]
    for row in cart_val.iter_rows(named=True)
}

In [23]:
len(true_items_val)

432

In [24]:
pop = (
    df_train
    .filter(pl.col("action_type") == "AT_CartUpdate")
    .group_by("product_id")
    .agg(pl.len().alias("n_carts"))
    .sort("n_carts", descending=True)
)
pop_items = pop.head(500)["product_id"].to_list()
print(f"pop_items: {len(pop_items)}")

pop_items: 500


In [25]:
user_ids = cart_train["user_id"].unique().to_list()
item_ids = cart_train["product_id"].unique().to_list()

In [26]:
user_to_idx = {uid: i for i, uid in enumerate(user_ids)}

item_to_idx = {iid: i for i, iid in enumerate(item_ids)}
idx_to_item = {idx: iid for iid, idx in item_to_idx.items()}

In [27]:
n_users = len(user_ids)
n_items = len(item_ids)

In [28]:
n_users, n_items

(2211, 13106)

In [29]:
rows = [user_to_idx[uid] for uid in cart_train["user_id"].to_list()]
cols = [item_to_idx[iid] for iid in cart_train["product_id"].to_list()]

data = np.ones((len(rows)), dtype=np.float32)

R = csr_matrix((data, (rows, cols)), shape=(n_users, n_items))

In [30]:
print(f"R.shape: {R.shape}")
print(f"R.nnz: {R.nnz}")
print(f"Dense: {R.nnz / (R.shape[0] * R.shape[1])}")

R.shape: (2211, 13106)
R.nnz: 132226
Dense: 0.004563078645588422


In [31]:
R[0,0]

np.float32(0.0)

## Ванильный iALS

In [32]:
model = AlternatingLeastSquares(
    factors=64,
    random_state=42
)

In [33]:
model.fit(R)

  0%|          | 0/15 [00:00<?, ?it/s]

In [34]:
model.user_factors.shape, model.item_factors.shape

((2211, 64), (13106, 64))

In [35]:
def build_als_candidates(
    model,
    R,
    user_ids_list: list,
    user_to_idx: dict,
    idx_to_item: dict,
    top_k: int = 500,
    filter_already_liked: bool = False
) -> dict:
    
    candidates = {}
    
    for uid in user_ids_list:
        if uid not in user_to_idx:
            continue
        user_idx = user_to_idx[uid]
        raw_index_candidates, scores = model.recommend(
            user_idx, R[user_idx], N=top_k, filter_already_liked_items=filter_already_liked
        )
        product_ids = [idx_to_item[idx] for idx in raw_index_candidates]
        candidates[uid] = product_ids
    return candidates

In [36]:
predicted_als = build_als_candidates(
    model,
    R,
    list(true_items_val.keys()),
    user_to_idx,
    idx_to_item,
    TOP_K
)

In [37]:
len(predicted_als)

398

In [38]:
# также как и раньше для cold-start users сделаем выдачу из популярных
fallback_count = 0
for user_id in true_items_val.keys():
    if user_id not in predicted_als or len(predicted_als[user_id]) == 0:
        predicted_als[user_id] = pop_items
        fallback_count += 1

In [39]:
result_als = evaluate_candidates(
    predicted_als,
    true_items_val,
    k=500,
    total_items=n_items
    
)

In [40]:
result_als

{'recall@500': 0.3828896456805354,
 'hit_rate@500': 0.8587962962962963,
 'mrr@500': 0.14697469021447168,
 'coverage': 0.6096444376621395,
 'n_requests': 432,
 'avg_candidates': 500.0}

## Эксперементы

### Подбираем размер эмбедов

In [41]:
results_by_factors = []

In [42]:
for factors in [64, 128, 256, 300, 400]:
    cur_model = AlternatingLeastSquares(
        factors=factors,
        random_state=42
    )
    cur_model.fit(R)

    cur_predicted = build_als_candidates(
        cur_model,
        R,
        list(true_items_val.keys()),
        user_to_idx,
        idx_to_item,
        top_k=500,
        filter_already_liked=False
    )
    
    # также для cold-start
    for user_id in true_items_val.keys():
        if user_id not in cur_predicted or len(cur_predicted[user_id]) == 0:
            cur_predicted[user_id] = pop_items
    
    cur_result = evaluate_candidates(
        cur_predicted,
        true_items_val,
        k=500,
        total_items=n_items,
    )
    
    cur_result["factors"] = factors
    results_by_factors.append(cur_result)
    
    print(f"  recall@500: {cur_result['recall@500']:.4f}")
    print(f"  mrr@500:    {cur_result['mrr@500']:.4f}")
    print(f"  coverage:   {cur_result['coverage']:.4f}")

  0%|          | 0/15 [00:00<?, ?it/s]

  recall@500: 0.3829
  mrr@500:    0.1470
  coverage:   0.6096


  0%|          | 0/15 [00:00<?, ?it/s]

  recall@500: 0.4099
  mrr@500:    0.1658
  coverage:   0.7412


  0%|          | 0/15 [00:00<?, ?it/s]

  recall@500: 0.4345
  mrr@500:    0.1657
  coverage:   0.8189


  0%|          | 0/15 [00:00<?, ?it/s]

  recall@500: 0.4414
  mrr@500:    0.1498
  coverage:   0.8245


  0%|          | 0/15 [00:00<?, ?it/s]

  recall@500: 0.4250
  mrr@500:    0.1457
  coverage:   0.8480


In [43]:
# результаты
df_results_factors = pl.DataFrame(results_by_factors)
df_results_factors

recall@500,hit_rate@500,mrr@500,coverage,n_requests,avg_candidates,factors
f64,f64,f64,f64,i64,f64,i64
0.38289,0.858796,0.146975,0.609644,432,500.0,64
0.409871,0.865741,0.165786,0.741187,432,500.0,128
0.434505,0.893519,0.165687,0.818862,432,500.0,256
0.441437,0.898148,0.149757,0.824508,432,500.0,300
0.42498,0.875,0.145746,0.848009,432,500.0,400


Сойдемся на emb_size=256 (при таком значении лучший Recall@k и MRR@k)

### Подбираем $\lambda$

In [44]:
regularization_list = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5]

In [45]:
results_by_reg = []

for reg in regularization_list:
    
    model = AlternatingLeastSquares(
        factors=256,
        regularization=reg,
        iterations=15,
        random_state=42,
    )
    model.fit(R)
    
    predicted = build_als_candidates(
        model,
        R,
        list(true_items_val.keys()),
        user_to_idx,
        idx_to_item,
        top_k=500,
        filter_already_liked=False,
    )
    
    for user_id in true_items_val.keys():
        if user_id not in predicted or len(predicted[user_id]) == 0:
            predicted[user_id] = pop_items
    
    result = evaluate_candidates(
        predicted,
        true_items_val,
        k=500,
        total_items=n_items,
    )
    
    result["regularization"] = reg
    results_by_reg.append(result)
    
    print(f"  recall@500: {result['recall@500']:.4f}")
    print(f"  mrr@500:    {result['mrr@500']:.4f}")
    print(f"  coverage:   {result['coverage']:.4f}")



  0%|          | 0/15 [00:00<?, ?it/s]

  recall@500: 0.4326
  mrr@500:    0.1654
  coverage:   0.8162


  0%|          | 0/15 [00:00<?, ?it/s]

  recall@500: 0.4340
  mrr@500:    0.1657
  coverage:   0.8179


  0%|          | 0/15 [00:00<?, ?it/s]

...

  recall@500: 0.4338
  mrr@500:    0.1673
  coverage:   0.8195


  0%|          | 0/15 [00:00<?, ?it/s]

  recall@500: 0.4350
  mrr@500:    0.1663
  coverage:   0.8155


  0%|          | 0/15 [00:00<?, ?it/s]

  recall@500: 0.4333
  mrr@500:    0.1662
  coverage:   0.8118


In [46]:
df_results_reg = pl.DataFrame(results_by_reg)
df_results_reg

recall@500,hit_rate@500,mrr@500,coverage,n_requests,avg_candidates,regularization
f64,f64,f64,f64,i64,f64,f64
0.432583,0.893519,0.165444,0.816191,432,500.0,0.001
0.434025,0.893519,0.165743,0.817946,432,500.0,0.005
0.434505,0.893519,0.165687,0.818862,432,500.0,0.01
0.433813,0.893519,0.167332,0.819548,432,500.0,0.05
0.434987,0.893519,0.166308,0.815504,432,500.0,0.1
0.433288,0.893519,0.166157,0.811766,432,500.0,0.5


Видим, что регуляризации практически не влияет на качество, оставим как у ванильного: 0.01

### Подберем парамет alpha - уверенность в позитивах

Напомнинание:

каждое значение в матрице R (в случае implicit ALS) считается как:

$$c_{ui} = 1 + \alpha \cdot R_{ui}$$

где $R_{ui} =$ 1 - если было взаимодействие, иначе 0

In [47]:
alpha_list = [1, 10, 40, 100, 500]

In [48]:
results_by_alpha = []

for alpha in alpha_list:
    
    model = AlternatingLeastSquares(
        factors=256,
        regularization=0.01,
        iterations=15,
        alpha=alpha,
        random_state=42,
    )
    model.fit(R)
    
    predicted = build_als_candidates(
        model,
        R,
        list(true_items_val.keys()),
        user_to_idx=user_to_idx,
        idx_to_item=idx_to_item,
        top_k=500,
        filter_already_liked=False,
    )
    
    for user_id in true_items_val.keys():
        if user_id not in predicted or len(predicted[user_id]) == 0:
            predicted[user_id] = pop_items
    
    result = evaluate_candidates(
        predicted,
        true_items_val,
        k=500,
        total_items=n_items,
    )
    
    result["alpha"] = alpha
    results_by_alpha.append(result)
    
    print(f"  recall@500: {result['recall@500']:.4f}")
    print(f"  hit_rate@500: {result['hit_rate@500']:.4f}")
    print(f"  mrr@500:    {result['mrr@500']:.4f}")
    print(f"  coverage:   {result['coverage']:.4f}")

  0%|          | 0/15 [00:00<?, ?it/s]

  recall@500: 0.4345
  hit_rate@500: 0.8935
  mrr@500:    0.1657
  coverage:   0.8189


  0%|          | 0/15 [00:00<?, ?it/s]

  recall@500: 0.4372
  hit_rate@500: 0.8958
  mrr@500:    0.1655
  coverage:   0.8612


  0%|          | 0/15 [00:00<?, ?it/s]

  recall@500: 0.4276
  hit_rate@500: 0.8819
  mrr@500:    0.1605
  coverage:   0.8802


  0%|          | 0/15 [00:00<?, ?it/s]

  recall@500: 0.4337
  hit_rate@500: 0.8843
  mrr@500:    0.1482
  coverage:   0.8842


  0%|          | 0/15 [00:00<?, ?it/s]

  recall@500: 0.4306
  hit_rate@500: 0.8819
  mrr@500:    0.1031
  coverage:   0.8601


In [49]:
df_results_alpha = pl.DataFrame(results_by_alpha)
df_results_alpha

recall@500,hit_rate@500,mrr@500,coverage,n_requests,avg_candidates,alpha
f64,f64,f64,f64,i64,f64,i64
0.434505,0.893519,0.165687,0.818862,432,500.0,1
0.437208,0.895833,0.165515,0.861209,432,500.0,10
0.42764,0.881944,0.16046,0.880208,432,500.0,40
0.433664,0.884259,0.148156,0.884175,432,500.0,100
0.430636,0.881944,0.103101,0.860064,432,500.0,500


Видим, что лучшая метрика recall@500 при $\alpha=10$, при этом с ростом параметра сильно просаживается mrr@500, хотя recall@500 не так сильно

In [50]:
BEST_ALS_PARAMS = {
    "factors": 256,
    "regularization": 0.01,
    "iterations": 50,
    "alpha": 10,
    "random_state": 42,
}

## Финальная модель

In [51]:
final_model_als = AlternatingLeastSquares(
    factors=BEST_ALS_PARAMS["factors"],
    regularization=BEST_ALS_PARAMS["regularization"],
    iterations=BEST_ALS_PARAMS["iterations"],
    alpha=BEST_ALS_PARAMS["alpha"],
    random_state=BEST_ALS_PARAMS["random_state"],
)

In [52]:
final_model_als.fit(R)

  0%|          | 0/50 [00:00<?, ?it/s]

In [53]:
final_predicted_als = build_als_candidates(
    final_model_als,
    R,
    list(true_items_val.keys()),
    user_to_idx=user_to_idx,
    idx_to_item=idx_to_item,
    top_k=500,
    filter_already_liked=False,
)

# также для cold-start
for user_id in true_items_val.keys():
    if user_id not in final_predicted_als or len(final_predicted_als[user_id]) == 0:
        final_predicted_als[user_id] = pop_items

In [54]:
len(final_predicted_als)

432

In [55]:
final_result_als = evaluate_candidates(
    final_predicted_als,
    true_items_val,
    k=500,
    total_items=n_items,
)

In [56]:
final_result_als

{'recall@500': 0.43595127886842056,
 'hit_rate@500': 0.8912037037037037,
 'mrr@500': 0.16769460717435217,
 'coverage': 0.8706699221730505,
 'n_requests': 432,
 'avg_candidates': 500.0}

In [57]:
final_model_als = AlternatingLeastSquares(
    factors=BEST_ALS_PARAMS["factors"],
    regularization=BEST_ALS_PARAMS["regularization"],
    iterations=BEST_ALS_PARAMS["iterations"],
    alpha=BEST_ALS_PARAMS["alpha"],
    random_state=BEST_ALS_PARAMS["random_state"],
)

In [58]:
final_model_als.fit(R)

  0%|          | 0/50 [00:00<?, ?it/s]

In [59]:
# сохраним финальную модель
Path("/kaggle/working/artifacts").mkdir(parents=True, exist_ok=True)

with open("/kaggle/working/artifacts/als_artifacts.pkl", "wb") as f:
    pickle.dump({
        "model": final_model_als,
        "user_to_idx": user_to_idx,
        "item_to_idx": item_to_idx,
        "idx_to_item": idx_to_item,
        "n_users": n_users,
        "n_items": n_items,
        "R": R,
        "pop_items": pop_items,
        "config": BEST_ALS_PARAMS,
    }, f)


In [61]:

print("ALS сохранен")
print(f"  Файл: als_artifacts.pkl")
print(f"  Модель: {type(final_model_als).__name__}")
print(f"  user_to_idx: {len(user_to_idx)}")
print(f"  item_to_idx: {len(item_to_idx)}")
print(f"  R: {R.shape}, nnz={R.nnz:,}")
print(f"  pop_items: {len(pop_items)}")

ALS сохранен
  Файл: als_artifacts.pkl
  Модель: AlternatingLeastSquares
  user_to_idx: 2211
  item_to_idx: 13106
  R: (2211, 13106), nnz=132,226
  pop_items: 500
